# 02 — Brain Tumor Segmentation with U-Net

Classification answers *"what kind of tumor is in this image?"* — segmentation answers *"which pixels are tumor?"*. This notebook trains a **U-Net** to produce a pixel-wise tumor mask on brain MRI.

**Why U-Net?** It is the de-facto baseline for biomedical segmentation. The encoder-decoder shape with **skip connections** lets it combine high-level semantics with fine-grained spatial detail (Ronneberger et al., MICCAI 2015).

**Audience note (AD researchers).** Substitute the LGG dataset with ADNI hippocampus masks (or any ROI you can label) and the *exact same* code segments brain regions for volumetry. Loss, metrics, augmentation discipline — all transfer.

**Dataset.** LGG MRI Segmentation by Mateusz Buda (Kaggle): 110 TCGA-LGG patients, FLAIR slices with binary tumor masks. We split **by patient** to avoid leakage.

## 0. Setup

```bash
pip install torch torchvision matplotlib numpy pillow kagglehub
```

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

import sys
from pathlib import Path

PROJ_DIR = Path.cwd().parent
sys.path.insert(0, str(PROJ_DIR))

import torch
import numpy as np
import matplotlib.pyplot as plt

from src.models import UNet, count_parameters
from src.data import get_device, get_segmentation_dataloaders
from src.train import (
    train_segmenter, evaluate_segmenter,
    dice_score, iou_score,
)
from src.viz import (
    plot_history,
    plot_segmentation_samples,
    plot_segmentation_predictions,
)

torch.manual_seed(42); np.random.seed(42)
device = get_device()
print(f"Using device: {device}")

## 1. Load LGG MRI segmentation data

Patient-level split: train ≈ 70%, val ≈ 15%, test ≈ 15% of patients. **No patient appears in more than one split.**

In [ ]:
IMG_SIZE = 128
BATCH_SIZE = 16

train_loader, val_loader, test_loader, (train_ds, val_ds, test_ds) = get_segmentation_dataloaders(
    data_dir=str(PROJ_DIR / "data"),
    batch_size=BATCH_SIZE,
    img_size=IMG_SIZE,
    augment=True,
)

X, M = next(iter(train_loader))
print(f"image batch : {tuple(X.shape)}  range: [{X.min():.2f}, {X.max():.2f}]")
print(f"mask  batch : {tuple(M.shape)}   unique: {torch.unique(M).tolist()}")

In [ ]:
# Visualize a few image-mask pairs
_ = plot_segmentation_samples(train_ds, n=4, title="LGG MRI: image / mask / overlay")
plt.show()

## 2. Define the U-Net model

Encoder: 4 down-sampling blocks (channels 32 → 64 → 128 → 256), bottleneck at 512.
Decoder: 4 transposed-conv up-samplers with **skip connections** from the matching encoder stage.
Output: a 1-channel logits map; apply `sigmoid` and threshold to get a binary mask.

In [ ]:
model = UNet(in_channels=1, out_channels=1, base=32)
print(f"Trainable params: {count_parameters(model):,}")

# Shape sanity check
dummy = torch.randn(2, 1, IMG_SIZE, IMG_SIZE)
out = model(dummy)
print(f"Input  : {tuple(dummy.shape)}")
print(f"Output : {tuple(out.shape)}   (logits, same H x W as input)")

## 3. Train

**Loss.** Combined `BCEWithLogitsLoss + (1 - DiceCoefficient)`. The Dice term helps with the heavy class imbalance (most pixels are background).

**Metrics.** Per-batch Dice and IoU (= Jaccard) for the foreground class.

In [ ]:
EPOCHS = 10
LR = 1e-3
WD = 1e-5

history = train_segmenter(
    model, train_loader, val_loader,
    epochs=EPOCHS, lr=LR, weight_decay=WD, device=device,
)

In [ ]:
_ = plot_history(
    history,
    left=("train_loss", "val_loss"),
    right=("train_dice", "val_dice"),
    right_scale=1.0,
    right_label="Dice coefficient",
)
plt.show()

## 4. Test-set evaluation

Report Dice and IoU on held-out patients.

In [ ]:
test_loss, test_dice, test_iou = evaluate_segmenter(model, test_loader, device)
print(f"[TEST] loss={test_loss:.4f}  dice={test_dice:.4f}  iou={test_iou:.4f}")

In [ ]:
# Side-by-side: input | GT mask | predicted mask | overlay (GT blue, pred red)
_ = plot_segmentation_predictions(model, test_ds, n=5, device=device, threshold=0.5)
plt.show()

## 5. Save the checkpoint

In [ ]:
out_dir = PROJ_DIR / "outputs"
out_dir.mkdir(exist_ok=True)
torch.save({
    "model_state": model.state_dict(),
    "history": history,
    "test_dice": test_dice,
    "test_iou": test_iou,
}, out_dir / "segment_unet.pt")
print("Saved.")

## 6. Hands-on exercises

### Architecture
1. **Skip connections matter.** Open `src/models.py:UNet.forward` and replace each skip concat (`torch.cat([up, e_i], dim=1)`) with just `up`. Retrain — how much does Dice drop? (This is the classic ablation that motivated U-Net.)
2. **Capacity.** Try `base=16` (lighter) vs `base=64` (heavier). Param count, GPU memory, and Dice?
3. **Activation.** Switch the decoder ReLU to GELU. Does it help on small data?

### Loss & metric
4. **Pure BCE vs. BCE+Dice.** In `src/train.py:bce_dice_loss`, set `bce_weight=1.0`. What happens to small tumors?
5. **Focal Tversky loss.** Replace `dice_loss` with `Tversky(alpha=0.7, beta=0.3)` and see if recall improves on small lesions.

### Data
6. **Subject leakage.** Switch from patient-level to slice-level random split (modify `_split_by_patient`) and re-run. The test Dice will jump unrealistically — that's the leakage you want to *prevent* in real research.
7. **Augmentation.** Add `random rotation ±15°` and `random crop+resize`. Does it help generalization on held-out patients?

### Translate to AD
8. **Hippocampus segmentation.** With ADNI subjects + FreeSurfer hippocampus masks, the only changes are: (a) 3D volumes → use `Conv3d` and an `nn.ConvTranspose3d`-based UNet, (b) cropping to a ROI around medial temporal lobe, (c) Dice computed in 3D. Sketch what `UNet3D` would look like.